# 🧠 Knowledge Distiller Agent
**Joyelearn Team | GDG Shanghai Gemma 4 Hackathon 2026**

**Track A: AI Agent**

This demo shows how Gemma 4 (26B MoE) acts as a Chief Knowledge Distillation Guide:
1. Conducts BEI (Behavioral Event Interview) to extract tacit knowledge
2. Uses Thinking Mode for deep reasoning
3. Calls native functions to save structured Skill Cards
4. Maintains clean conversation history (thought content filtered out)


In [ ]:
# Step 1: Install dependencies
!pip install google-genai -q

In [ ]:
# Step 2: Configure API Key
import os
os.environ['GEMINI_API_KEY'] = 'YOUR_API_KEY_HERE'  # Replace with your key

In [ ]:
# Step 3: Load agent (copy of agent.py)
from google import genai
from google.genai import types

MODEL = 'models/gemma-4-26b-a4b-it'
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
skill_library = {}

SYSTEM_PROMPT = """You are the Chief Knowledge Distillation Guide, built by Joyelearn.
Your mission: extract tacit knowledge from domain experts through structured interviews,
then distill it into reusable organizational assets (SOPs, Skill Cards).
Use BEI (Behavioral Event Interview) method. Ask only ONE question per turn.
When enough context is gathered, offer to generate a Skill Card."""

TOOLS = [
    types.Tool(function_declarations=[
        types.FunctionDeclaration(
            name='save_skill_card',
            description='Call ONLY when interview is complete and user confirms to save. NOT during active interview.',
            parameters={
                'type': 'object',
                'properties': {
                    'skill_name': {'type': 'string'},
                    'applicable_scenarios': {'type': 'array', 'items': {'type': 'string'}},
                    'core_steps': {'type': 'array', 'items': {
                        'type': 'object',
                        'properties': {
                            'step': {'type': 'integer'},
                            'action': {'type': 'string'},
                            'key_decision': {'type': 'string'}
                        }
                    }},
                    'underlying_principle': {'type': 'string'}
                },
                'required': ['skill_name', 'applicable_scenarios', 'core_steps', 'underlying_principle']
            }
        )
    ])
]

def extract_answer(response):
    """Filter thought blocks — IRON RULE per Gemma 4 spec"""
    parts = response.candidates[0].content.parts
    return ''.join(p.text for p in parts if not p.thought and p.text).strip()

def chat(history, user_message):
    history.append({'role': 'user', 'parts': [{'text': user_message}]})
    response = client.models.generate_content(
        model=MODEL,
        contents=history,
        config=types.GenerateContentConfig(
            temperature=1.0, top_p=0.95, top_k=64,
            max_output_tokens=4096,
            tools=TOOLS,
            system_instruction=SYSTEM_PROMPT
        )
    )
    parts = response.candidates[0].content.parts
    tool_calls = [p for p in parts if hasattr(p, 'function_call') and p.function_call]
    if tool_calls:
        for p in tool_calls:
            fc = p.function_call
            skill_id = f'SKILL-{len(skill_library)+1:03d}'
            skill_library[skill_id] = dict(fc.args)
            print(f'[Tool] {fc.name} called → Skill saved as {skill_id}')
            print(f'[Skill Card]')
            print(json.dumps(dict(fc.args), indent=2, ensure_ascii=False))
    answer = extract_answer(response)
    history.append({'role': 'model', 'parts': [{'text': answer}]})
    return answer

import json
print('Agent ready. Model:', MODEL)

## Demo: Full Interview Session
Simulating a real knowledge extraction interview with a Sales Director

In [ ]:
# Turn 1: Start interview
history = []
r = chat(history, "I want to extract our Sales Director's negotiation experience with large enterprise clients.")
print('Guide:', r)

In [ ]:
# Turn 2: Provide context
r = chat(history, "A state-owned enterprise client, contract value 8 million, procurement kept pushing for discounts. We closed at full price.")
print('Guide:', r)

In [ ]:
# Turn 3: Share the key moment
r = chat(history, "The turning point was when the director stopped defending the price and instead asked the client: 'What happens to your project if this procurement fails?' That reframed the whole conversation.")
print('Guide:', r)

In [ ]:
# Turn 4: Save as Skill Card
r = chat(history, "Great, please save this as a skill card now.")
print('Guide:', r)

In [ ]:
# Show saved skill library
print(f'\nSkill Library ({len(skill_library)} skills saved):')
for sid, skill in skill_library.items():
    print(f'\n{sid}:', json.dumps(skill, indent=2, ensure_ascii=False))